# pymysql

## Example 1: escape values the way the driver does before sending a query

In [ ]:
# Example 1: converters.escape_item quotes strings, renders numbers, None and dates as SQL literals; sequences go
# through Connection.escape(), which reads the server's SQL mode (PyMySQL 1.2.4, GHSA-m75w-33c4-7v8q)
import datetime
import pymysql
from pymysql.constants import SERVER_STATUS
from pymysql.converters import escape_item, escape_string

assert escape_string("it's") == "it\\'s"
assert escape_item("O'Hara", "utf8mb4") == "'O\\'Hara'"
assert escape_item(None, "utf8mb4") == "NULL" and escape_item(3.5, "utf8mb4") == "3.5e0"
assert escape_item(datetime.date(2026, 11, 1), "utf8mb4") == "'2026-11-01'"
try:
    escape_item((1, "a"), "utf8mb4")
except TypeError:
    pass  # 1.2.4: the default converter mapping no longer escapes tuples, lists or sets
else:
    raise AssertionError("escape_item still escapes a tuple: the 1.2.4 SQL-injection fix is missing")
conn = pymysql.connect(charset="utf8mb4", defer_connect=True)
conn.server_status = 0  # what the server reports at login; no server here
assert conn.escape((1, "it's")) == "(1,'it\\'s')"
conn.server_status = SERVER_STATUS.SERVER_STATUS_NO_BACKSLASH_ESCAPES  # the SQL mode the fix is about
assert conn.escape((1, "it's")) == "(1,'it''s')"
print("Example 1:", escape_item("O'Hara", "utf8mb4"), conn.escape((1, "it's")))

## Example 2: a connection object built without connecting

## Example 3: connecting to a port with no server is refused cleanly

In [ ]:
# Example 3: a closed local port gives OperationalError 2003 ("Can't connect to MySQL server"), not a hang
import socket
import pymysql

with socket.socket() as probe:
    probe.bind(("127.0.0.1", 0))
    closed_port = probe.getsockname()[1]
try:
    pymysql.connect(host="127.0.0.1", port=closed_port, user="x", connect_timeout=5)
except pymysql.err.OperationalError as error:
    assert error.args[0] == 2003, error.args
    print("Example 3: refused as expected:", error.args[0], str(error.args[1])[:60])
else:
    raise AssertionError("a closed port accepted a MySQL connection")

## Example 4: the DB-API 2.0 module surface

In [ ]:
# Example 4: PyMySQL exposes the DB-API globals and error hierarchy
import pymysql

assert pymysql.apilevel == "2.0" and pymysql.paramstyle == "pyformat" and pymysql.threadsafety == 1
assert issubclass(pymysql.err.IntegrityError, pymysql.err.DatabaseError)
assert issubclass(pymysql.err.DatabaseError, pymysql.err.Error)
assert pymysql.Binary(b"\x00\x01") == b"\x00\x01"
print("Example 4: apilevel", pymysql.apilevel, "paramstyle", pymysql.paramstyle, "version", pymysql.__version__)

## Example 5: the password scrambles behind the auth plugins

In [ ]:
# Example 5: mysql_native_password (SHA1), caching_sha2_password (SHA256) and ed25519 (PyNaCl) scrambles
from pymysql import _auth

nonce = bytes(range(20))
native = _auth.scramble_native_password(b"secret", nonce)
sha2 = _auth.scramble_caching_sha2(b"secret", nonce)
ed = _auth.ed25519_password(b"secret", nonce + bytes(12))
assert len(native) == 20 and len(sha2) == 32 and len(ed) == 64
assert native == _auth.scramble_native_password(b"secret", nonce) and native != _auth.scramble_native_password(b"other", nonce)
print("Example 5: native", native.hex()[:16], "sha2", sha2.hex()[:16], "ed25519", ed.hex()[:16])